https://auto.gluon.ai/stable/tutorials/tabular/advanced/tabular-custom-model.html

In [1]:
from __future__ import annotations

from autogluon.core.data import LabelCleaner
from autogluon.core.models import AbstractModel
from autogluon.core.utils import infer_problem_type
from autogluon.features.generators import AutoMLPipelineFeatureGenerator, LabelEncoderFeatureGenerator
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

c:\Users\Bo_wo\Desktop\code\epita\pfee-epita-qml-tde\docs\QSVM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SVMModel(AbstractModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._feature_generator = None

    def _preprocess(self, X, **kwargs):
        X = super()._preprocess(X, **kwargs)
        if self._feature_generator is None:
            self._feature_generator = LabelEncoderFeatureGenerator(verbosity=0)
            self._feature_generator.fit(X=X)
        if self._feature_generator.features_in:
            X = X.copy()
            X[self._feature_generator.features_in] = self._feature_generator.transform(X=X)
        return X.fillna(0)

    def _fit(self, X, y, **kwargs) -> None: # type: ignore[override]
        X = self.preprocess(X)
        params = self._get_model_params()
        self.model = make_pipeline(
            StandardScaler(),
            CalibratedClassifierCV(LinearSVC(**params)),
        )
        self.model.fit(X, y)

    def _set_default_params(self):
        defaults = {
            "random_state": 0,
            "tol": 1e-5,
            "max_iter": 2000,
        }
        for param, val in defaults.items():
            self._set_default_param_value(param, val)

    def _get_default_auxiliary_params(self) -> dict:
        default_auxiliary_params = super()._get_default_auxiliary_params()
        default_auxiliary_params.update({"valid_raw_types": ["int", "float", "category"]})
        return default_auxiliary_params

In [3]:
def standalone_fit(train_data, test_data, label):
    X      = train_data.drop(columns=[label])
    y      = train_data[label]
    X_test = test_data.drop(columns=[label])
    y_test = test_data[label]

    problem_type  = infer_problem_type(y=y)
    label_cleaner = LabelCleaner.construct(problem_type=problem_type, y=y)
    y_clean       = label_cleaner.transform(y)
    y_test_clean  = label_cleaner.transform(y_test)

    feature_generator = AutoMLPipelineFeatureGenerator()
    X_clean      = feature_generator.fit_transform(X)
    X_test_clean = feature_generator.transform(X_test)

    model = SVMModel()
    model.fit(X=X_clean, y=y_clean)

    score = model.score(X_test_clean, y_test_clean.to_numpy())

    assert model.eval_metric is not None
    print(f"Standalone test score ({model.eval_metric.name}): {score:.4f}")

    """
    y_pred_orig = label_cleaner.inverse_transform(model.predict(X_test_clean))
    print(y_pred_orig.head())
    """

def tabular_predictor(train_data, test_data, label):
    predictor = TabularPredictor(label=label).fit(
        train_data,
        hyperparameters={SVMModel: {}},
    )

    print(predictor.evaluate(test_data))
    print(predictor.leaderboard(test_data))

In [4]:
train_data = TabularDataset("https://autogluon.s3.amazonaws.com/datasets/Inc/train.csv")
test_data  = TabularDataset("https://autogluon.s3.amazonaws.com/datasets/Inc/test.csv")
label = "class"

train_data = train_data.sample(n=1000, random_state=0)

# standalone_fit(train_data, test_data, label)
tabular_predictor(train_data, test_data, label)

No path specified. Models will be saved in: "AutogluonModels\ag-20260529_111749"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.5
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       11.31 GB / 31.71 GB (35.7%)
Disk Space Avail:   138.91 GB / 446.28 GB (31.1%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mit

{'accuracy': 0.8207595455010749, 'balanced_accuracy': np.float64(0.6841205221250508), 'mcc': 0.44722348827768, 'roc_auc': np.float64(0.8503609836783523), 'f1': 0.5289211729889696, 'precision': 0.7026447462473195, 'recall': 0.4240724762726488}
                 model  score_test  score_val eval_metric  pred_time_test  \
0             SVMModel     0.82076       0.81    accuracy        0.007897   
1  WeightedEnsemble_L2     0.82076       0.81    accuracy        0.009065   

   pred_time_val  fit_time  pred_time_test_marginal  pred_time_val_marginal  \
0       0.003624  0.027440                 0.007897                0.003624   
1       0.004277  0.030233                 0.001168                0.000653   

   fit_time_marginal  stack_level  can_infer  fit_order  
0           0.027440            1       True          1  
1           0.002794            2       True          2  
